In [1]:
import sys, time, gc
from pathlib import Path
import duckdb, pandas as pd

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import src.blocking as B, src.rerank as R, src.features as F

ARTIFACTS = ROOT / "artifacts"
PARQ, KEYS2 = ARTIFACTS / "parquet", ARTIFACTS / "keys_v2"
def pq(name): return f"read_parquet('{(PARQ / f'{name}.parquet').as_posix()}')"
def k2(name): return f"read_parquet('{(KEYS2 / f'{name}.parquet').as_posix()}')"

# separate DB file + temp dir: no lock conflict with the training notebook
con = duckdb.connect((ARTIFACTS / "test.duckdb").as_posix())
(ARTIFACTS / "duckdb_tmp_test").mkdir(exist_ok=True)
B.setup(con, threads=14, memory_limit="9GB", temp_dir=(ARTIFACTS / "duckdb_tmp_test").as_posix())
F.setup_macros(con)

t0 = time.time()
for src_name, tbl in [("test_source1", "k_t1"), ("test_source2", "k_t2"), ("test_source3", "k_t3")]:
    con.execute(f"CREATE OR REPLACE TABLE {tbl} AS SELECT * FROM {k2(src_name)}")
S1_T, TGT_PARTS_T = "k_t1", ["k_t2", "k_t3"]
TGT_T = "(SELECT * FROM k_t2 UNION ALL SELECT * FROM k_t3)"
print(f"materialized {time.time() - t0:.0f}s")

# blocking + cycle-3 reranker + cut, exactly as for train
reranker = R.load(ARTIFACTS / "rerank_lgb_c3.txt")
con.execute("CREATE OR REPLACE TABLE t_cut (s1_id BIGINT, tgt_id BIGINT, src BIGINT, p DOUBLE, rk2 BIGINT)")
t0 = time.time()
B.run_blocking(con, "t_cut", S1_T, TGT_PARTS_T, s1_ids_tbl=None, k_max=50, s1_chunks=64,
               on_chunk=lambda tbl: R.rerank_and_cut(con, tbl, S1_T, TGT_T, reranker, "t_cut", tau=0.003, cap=8))
print(f"test blocking done in {(time.time() - t0) / 60:.1f} min")

# everything the matcher needs on the test side (IDF over the TEST target pool, as for train)
F.build_rev(con, "t_cut", "t_rev")
F.build_name_idf(con, TGT_T)
F.build_addr_idf(con, TGT_T)
F.build_raw_names(con, pq("test_source1"),
                  f"(SELECT * FROM {pq('test_source2')} UNION ALL SELECT * FROM {pq('test_source3')})")
con.execute("CHECKPOINT")

display(con.execute("""
SELECT k.country_key, count(*) AS n_s1, avg(coalesce(c.nc, 0)) AS avg_candidates
FROM (SELECT eid(entity_id) AS s1_id, country_key FROM k_t1) k
LEFT JOIN (SELECT s1_id, count(*) AS nc FROM t_cut GROUP BY 1) c USING (s1_id)
GROUP BY 1 ORDER BY 1""").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

materialized 28s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,country_key,n_s1,avg_candidates
0,france,259452,12.264153
1,india,809986,7.309394
2,us,663106,7.042378


In [2]:
import sys, time, gc, subprocess, importlib
import xgboost as xgb
import src.features as F
import src.predict as P
importlib.reload(F); importlib.reload(P)
F.setup_macros(con)
con.execute("PRAGMA threads=14"); con.execute("SET memory_limit='9GB'")

# v3 features don't use tok_odds, but the feature SQL joins it -> make sure it exists
con.execute("CREATE TABLE IF NOT EXISTS tok_odds (kind VARCHAR, tok VARCHAR, lo DOUBLE, n BIGINT)")

# final matcher: v3 features, 600k training S1 (validation macro-F0.5 0.9802)
m = xgb.XGBClassifier()
m.load_model(str(ARTIFACTS / "matcher_xgb_v3_c3_big.json"))

# ---- 1. matcher probabilities for every test candidate ----
t0 = time.time()
con.execute("CREATE OR REPLACE TABLE t_q (s1_id BIGINT, tgt_id BIGINT, q DOUBLE)")
NCH = 32
for j in range(NCH):
    con.execute(f"CREATE OR REPLACE TABLE _sc AS SELECT * FROM t_rev WHERE hash(s1_id) % {NCH} = {j}")
    df = F.build_features(con, "_sc", S1_T, TGT_T)
    if len(df):
        df["q"] = m.predict_proba(df[F.FEATS_V3].astype("float32"))[:, 1]
        con.register("_q", df[["s1_id", "tgt_id", "q"]])
        con.execute("INSERT INTO t_q SELECT * FROM _q")
        con.unregister("_q")
    del df
    gc.collect()
    if (j + 1) % 4 == 0:
        print(f"   scored {j + 1}/{NCH}  {(time.time() - t0) / 60:.1f} min", flush=True)
con.execute("DROP TABLE IF EXISTS _sc")
print("t_q rows:", con.execute("SELECT count(*) FROM t_q").fetchone()[0])

# ---- 2. decision: q >= 0.7 + one-to-many, extra matches of the same S1 need q >= 0.8 ----
P.decide(con, "t_q", "t_pred", 0.7, one_to_many=True)
con.execute("""
CREATE OR REPLACE TABLE t_pred AS
SELECT s1_id, tgt_id, q FROM (
    SELECT *, row_number() OVER (PARTITION BY s1_id ORDER BY q DESC) AS rk FROM t_pred
) WHERE rk = 1 OR q >= 0.8""")

# ---- 3. write both files + checks ----
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)
cand = P.write_submission(con, "t_cut",  S1_T, TGT_T, "candidate_entity_ids", OUT / "candidate_pairs.tsv")
pred = P.write_submission(con, "t_pred", S1_T, TGT_T, "matched_entity_ids",   OUT / "matching_results.tsv")

n = con.execute(f"SELECT count(*) FROM {S1_T}").fetchone()[0]
assert len(cand) == len(pred) == n, (len(cand), len(pred), n)
assert pred.source1_entity_id.is_unique
nm = pred.matched_entity_ids.str.count(",") + (pred.matched_entity_ids != "")
nc = cand.candidate_entity_ids.str.count(",") + (cand.candidate_entity_ids != "")
print(f"rows {n:,} | avg candidates {nc.mean():.2f} | avg matches {nm.mean():.2f} "
      f"| empty {(nm == 0).mean():.3f} | total {(time.time() - t0) / 60:.1f} min")

# per-country sanity
cty = con.execute(f"SELECT entity_id AS source1_entity_id, country_key FROM {S1_T}").fetchdf()
chk = pred.assign(n_match=nm).merge(cty, on="source1_entity_id")
display(chk.groupby("country_key").agg(n=("n_match", "size"), avg_matches=("n_match", "mean"),
                                       empty_rate=("n_match", lambda s: (s == 0).mean())).round(3))

# ---- 4. official validator ----
r = subprocess.run([sys.executable, str(ROOT / "utils" / "validate_submission.py"),
                    "--matching", str(OUT / "matching_results.tsv"),
                    "--candidate", str(OUT / "candidate_pairs.tsv"),
                    "--test-dir", str(ROOT / "dataset" / "test")],
                   capture_output=True, text=True)
print(r.stdout[-1000:], r.stderr[-500:])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   scored 4/32  4.9 min


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows 1,732,544 | avg candidates 7.95 | avg matches 3.36 | empty 0.057 | total 47.3 min


,n,avg_matches,empty_rate
country_key,,,
france,259452,3.388,0.052
india,809986,3.322,0.058
us,663106,3.385,0.057


ML Challenge 2026 — submission validator
  test dir: D:\ML Challenge\6ab10eb3b23ba_student_resource\student_resource\dataset\test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (98747 empty, 1633797 non-empty).
  candidate_pairs.tsv: 1732544 rows (244 empty, 1732300 non-empty).

PASS — no blocking issues found. Safe to submit.
 


In [3]:
CTY = "(SELECT eid(entity_id) AS s1_id, country_key FROM k_t1)"

def predict_rule(t1, t2, out):
    P.decide(con, "t_q", out, t1, one_to_many=True)
    con.execute(f"""CREATE OR REPLACE TABLE {out} AS SELECT s1_id, tgt_id, q FROM (
        SELECT *, row_number() OVER (PARTITION BY s1_id ORDER BY q DESC) AS rk FROM {out})
        WHERE rk = 1 OR q >= {t2}""")

rows = []
for t1, t2 in [(0.5, 0.7), (0.6, 0.75), (0.7, 0.8), (0.8, 0.85), (0.85, 0.9), (0.9, 0.95)]:
    predict_rule(t1, t2, "tp_tmp")
    rows.append(con.execute(f"""
        SELECT '{t1}/{t2}' AS rule, k.country_key,
               avg(coalesce(p.n, 0)) AS avg_matches, avg((coalesce(p.n, 0) = 0)::INT) AS empty_rate
        FROM {CTY} k LEFT JOIN (SELECT s1_id, count(*) AS n FROM tp_tmp GROUP BY 1) p USING (s1_id)
        GROUP BY ALL ORDER BY ALL""").fetchdf())
display(pd.concat(rows).pivot(index="rule", columns="country_key", values=["avg_matches", "empty_rate"]).round(3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

avg_matches               empty_rate              
country_key      france  india     us     france  india     us
rule                                                          
0.5/0.7           3.431  3.356  3.415      0.048  0.055  0.056
0.6/0.75          3.411  3.340  3.400      0.050  0.057  0.057
0.7/0.8           3.388  3.322  3.385      0.052  0.058  0.057
0.8/0.85          3.360  3.300  3.368      0.054  0.060  0.059
0.85/0.9          3.324  3.273  3.345      0.056  0.061  0.059
0.9/0.95          3.258  3.224  3.307      0.058  0.063  0.060